# 01. 데이터 준비

## 이 노트북이 하는 일
1. 한국 LEED 건물 460개 PDF 스코어카드 로드
2. CSV 프로젝트 디렉토리와 매칭
3. Rule 기반으로 v5 스키마로 표준화
4. parquet 저장

## 출력 파일
- `data/processed/project_features.parquet` (460건 × 28컬럼)
- `data/processed/standardized_credits.parquet` (9,747건 크레딧 레벨)

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트를 PYTHONPATH에 추가
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.langgraph_workflow.graph import run_standardization
from src.data.loader import LEEDDataLoader

print(f'작업 디렉토리: {ROOT}')

## 1단계: 프로젝트 디렉토리 로딩

USGBC Public LEED Project Directory CSV 로드.

In [ ]:
loader = LEEDDataLoader()
csv_df = loader.load_project_directory()
print(f'CSV 프로젝트: {len(csv_df)}개')
csv_df.head()

## 2단계: PDF 파일 목록 확인

In [ ]:
SCORECARD_DIR = ROOT / 'data' / 'raw' / 'scorecards'
pdf_files = sorted(SCORECARD_DIR.glob('*.pdf'))
print(f'PDF 파일: {len(pdf_files)}개')

## 3단계: 전체 파이프라인 실행

각 PDF에 대해:
1. PDF 파싱 → 2. CSV 매칭 → 3. Rule 매핑 → 4. 수학 검증 → 5. 저장

460건 기준 약 5분 소요. 진행률은 50건마다 표시.

In [ ]:
def build_credit_rows(project_id, version, leed_system, credit_mappings):
    """크레딧 매핑 → dict 리스트"""
    rows = []
    for cm in credit_mappings:
        rows.append({
            'project_id': project_id,
            'source_version': version,
            'leed_system': leed_system,
            'source_credit_name': cm.get('credit_name', ''),
            'v5_credit_code': cm.get('v5_code', 'UNKNOWN'),
            'v5_category': cm.get('v5_category'),
            'points_awarded': cm.get('awarded', 0),
            'points_possible': cm.get('possible', 0),
            'mapping_method': 'rule' if cm.get('matched') else 'unmatched',
            'confidence': cm.get('confidence'),
        })
    return rows

feature_rows, credit_rows = [], []
counters = {'success': 0, 'failed': 0, 'rule': 0, 'llm': 0}

for i, pdf in enumerate(pdf_files, 1):
    if i % 50 == 0 or i == 1 or i == len(pdf_files):
        print(f'[{i:3d}/{len(pdf_files)}] {pdf.name[:55]}')
    try:
        state = run_standardization(pdf_path=str(pdf), directory_df=csv_df)
        if state.get('status') != 'completed':
            raise RuntimeError(f"status={state.get('status')}")
        
        final = state['final_v5_data']
        project = state.get('project', {})
        rule_result = state.get('rule_mapping_result', {})
        math_result = state.get('math_validation_result', {})
        
        version = final.get('original_version', 'unknown')
        project_id = final.get('project_id', pdf.name)
        leed_system = final.get('leed_system', '')
        track = final.get('standardization_track', 'rule')
        
        feature_rows.append({
            'project_id': project_id,
            'project_name': final.get('project_name', ''),
            'leed_system': leed_system,
            'building_type': final.get('building_type', ''),
            'gross_area_sqm': final.get('gross_area_sqm', 0),
            'original_version': version,
            'certification_level': final.get('certification_level', ''),
            'total_score_original': final.get('total_score_original', 0),
            'total_score_v5': final.get('total_score_v5', 0),
            'achievement_ratio_original': final.get('achievement_ratio_original', 0),
            'achievement_ratio_v5': final.get('achievement_ratio_v5', 0),
            'standardization_track': track,
            'drift': math_result.get('ratio_drift', 0),
            'credit_rule_hit_rate': rule_result.get('credit_rule_hit_rate', None),
            **{k: v for k, v in final.items() if k.startswith('ratio_')},
            **{k: v for k, v in final.items() if k.startswith('score_v5_')},
        })
        
        cm = rule_result.get('credit_mappings', [])
        if cm:
            credit_rows.extend(build_credit_rows(project_id, version, leed_system, cm))
        else:
            # v2009: 카테고리 합계로 대체
            for cat, score in rule_result.get('mapped_categories', {}).items():
                credit_rows.append({
                    'project_id': project_id,
                    'source_version': version,
                    'leed_system': leed_system,
                    'source_credit_name': f'[category] {cat}',
                    'v5_credit_code': f'CAT_{cat}',
                    'v5_category': cat,
                    'points_awarded': project.get('categories', {}).get(cat, 0),
                    'points_possible': project.get('categories_possible', {}).get(cat, 0),
                    'mapping_method': 'category_proportional',
                    'confidence': 'high',
                })
        
        counters['success'] += 1
        counters[track] = counters.get(track, 0) + 1
    except Exception as e:
        counters['failed'] += 1

print(f"\n완료: {counters}")

## 4단계: parquet 저장

In [ ]:
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

feat_df = pd.DataFrame(feature_rows)
cred_df = pd.DataFrame(credit_rows)

feat_df.to_parquet(PROCESSED / 'project_features.parquet', index=False)
cred_df.to_parquet(PROCESSED / 'standardized_credits.parquet', index=False)

print(f'project_features.parquet: {len(feat_df)}행')
print(f'standardized_credits.parquet: {len(cred_df)}행')
print(f"\n버전 분포: {feat_df['original_version'].value_counts().to_dict()}")
print(f"등급 분포: {feat_df['certification_level'].value_counts().to_dict()}")

## 5단계: 결과 검증

In [ ]:
# v5 만점 기준으로 ratio 값이 1 넘지 않는지 체크
ratio_cols = [c for c in feat_df.columns if c.startswith('ratio_') and c != 'ratio_original']
over_max = 0
for col in ratio_cols:
    over_max += (feat_df[col] > 1.01).sum()
print(f'ratio > 1.01 위반: {over_max}건 (0이어야 정상)')

# drift 분포
print(f"\n평균 drift: {feat_df['drift'].mean()*100:.2f}%")
print(f"drift > 20%: {(feat_df['drift'] > 0.20).sum()}건")

# 크레딧 매핑 방식
print(f"\n크레딧 매핑 방식:")
print(cred_df['mapping_method'].value_counts())

## 다음 단계

- `02_LLM_검증.ipynb`: LLM 전문가 리뷰 (표본)
- `03_SHAP_분석.ipynb`: XGBoost + SHAP 분석